# Week 4 Exercise — Python to C++ Code Converter

Use a frontier model to generate **high-performance C++ code** from Python code (same idea as the course Day 3–5 code generator). Paste Python, pick a model (OpenRouter or Ollama), and get equivalent C++.

Set `OPENROUTER_API_KEY` in `.env`. For Ollama: `ollama pull llama3.2` (or a coder model) and have Ollama running.

In [ ]:
import os
import re
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)
api_key = os.getenv("OPENROUTER_API_KEY")

if not api_key:
    print("Add OPENROUTER_API_KEY to .env")
else:
    print("OpenRouter API key loaded.")

MODELS = {
    "OpenRouter: gpt-4o-mini": ("openai/gpt-4o-mini", "https://openrouter.ai/api/v1", api_key),
    "OpenRouter: claude-3-haiku": ("anthropic/claude-3-haiku", "https://openrouter.ai/api/v1", api_key),
    "Ollama: llama3.2": ("llama3.2", "http://localhost:11434/v1", "ollama"),
}

def get_client(model_key):
    model_id, base_url, key = MODELS.get(model_key, list(MODELS.values())[0])
    return OpenAI(api_key=key or "ollama", base_url=base_url), model_id

In [ ]:
SYSTEM_PROMPT = """Your task is to convert Python code into high-performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ must produce identical output to the Python when run. Use C++17 or later.
No markdown fences—output raw C++ only."""

In [ ]:
def python_to_cpp(python_code: str, model_key: str) -> str:
    if not python_code or not python_code.strip():
        return "Paste Python code above and click Convert."
    client, model_id = get_client(model_key)
    user_msg = f"""Port this Python code to C++ with a fast implementation that produces identical output.
Respond only with C++ code, no explanation.

```python
{python_code.strip()}
```
"""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg}
    ]
    try:
        r = client.chat.completions.create(model=model_id, messages=messages)
        text = (r.choices[0].message.content or "").strip()
        text = re.sub(r"^```(?:cpp|c\+\+)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
        return text.strip()
    except Exception as e:
        return f"Error: {e}"

In [ ]:
sample_python = """def factorial(n):
    if n <= 1:
        return 1
    return n * factorial(n - 1)

for i in range(6):
    print(f"factorial({i}) = {factorial(i)}")
"""

with gr.Blocks(title="Python to C++") as demo:
    gr.Markdown("# Week 4 — Python to C++ Code Converter")
    with gr.Row():
        py_in = gr.Code(label="Python code", value=sample_python, language="python", lines=14)
        cpp_out = gr.Code(label="Generated C++", value="", language="cpp", lines=14)
    with gr.Row():
        model_dd = gr.Dropdown(choices=list(MODELS.keys()), value=list(MODELS.keys())[0], label="Model")
        btn = gr.Button("Convert")
    btn.click(fn=python_to_cpp, inputs=[py_in, model_dd], outputs=[cpp_out])
    gr.Examples(
        [[sample_python, list(MODELS.keys())[0]]],
        inputs=[py_in, model_dd],
        label="Example"
    )
demo.launch()